In [41]:
import pandas as pd
import sqlite3 as sqlite3

sample = pd.read_csv("../data/raw/bikes_2026_01.csv").head()
sample

,system_id,last_reported,station_id,num_bikes_available,num_docks_available,is_installed,is_renting,is_returning,name,short_name,address,lat,lon,region_id,capacity
0,dublin_bikes,2026-01-01 00:05:00,1,24,7,True,True,True,CLARENDON ROW,NaN,Clarendon Row,53.340927,-6.262501,NaN,31
1,dublin_bikes,2026-01-01 00:05:00,10,16,0,True,True,True,DAME STREET,NaN,Dame Street,53.344006,-6.266802,NaN,16
2,dublin_bikes,2026-01-01 00:05:00,101,5,25,True,True,True,KING STREET NORTH,NaN,King Street North,53.350292,-6.273507,NaN,30
3,dublin_bikes,2026-01-01 00:05:00,103,5,35,True,True,True,GRANGEGORMAN LOWER (SOUTH),NaN,Grangegorman Lower (South),53.354664,-6.278681,NaN,40
4,dublin_bikes,2026-01-01 00:05:00,104,20,20,True,True,True,GRANGEGORMAN LOWER (CENTRAL),NaN,Grangegorman Lower (Central),53.355170,-6.278424,NaN,40


In [42]:
bike_files = ["../data/raw/bikes_2026_01.csv", "../data/raw/bikes_2026_02.csv", "../data/raw/bikes_2026_03.csv",
              "../data/raw/bikes_2026_04.csv", "../data/raw/bikes_2026_05.csv", "../data/raw/bikes_2026_06.csv"]

len(bike_files)

6

In [43]:
sample = pd.read_csv(bike_files[0]).head()
print(sample.dtypes)
sample

system_id                  str
last_reported              str
station_id               int64
num_bikes_available      int64
num_docks_available      int64
is_installed              bool
is_renting                bool
is_returning              bool
name                       str
short_name             float64
address                    str
lat                    float64
lon                    float64
region_id              float64
capacity                 int64
dtype: object


,system_id,last_reported,station_id,num_bikes_available,num_docks_available,is_installed,is_renting,is_returning,name,short_name,address,lat,lon,region_id,capacity
0,dublin_bikes,2026-01-01 00:05:00,1,24,7,True,True,True,CLARENDON ROW,NaN,Clarendon Row,53.340927,-6.262501,NaN,31
1,dublin_bikes,2026-01-01 00:05:00,10,16,0,True,True,True,DAME STREET,NaN,Dame Street,53.344006,-6.266802,NaN,16
2,dublin_bikes,2026-01-01 00:05:00,101,5,25,True,True,True,KING STREET NORTH,NaN,King Street North,53.350292,-6.273507,NaN,30
3,dublin_bikes,2026-01-01 00:05:00,103,5,35,True,True,True,GRANGEGORMAN LOWER (SOUTH),NaN,Grangegorman Lower (South),53.354664,-6.278681,NaN,40
4,dublin_bikes,2026-01-01 00:05:00,104,20,20,True,True,True,GRANGEGORMAN LOWER (CENTRAL),NaN,Grangegorman Lower (Central),53.355170,-6.278424,NaN,40


## funcao de limpeza
seleciona as colunas, converte bool pra 0/1 e cria as duas colunas de tempo.
funcoes: astype, dt.tz_localize, dt.tz_convert
utc: continuo, bom pra agrupar. local: a hora que a pessoa ve no relogio

In [44]:
def clean_chunk(chunk):
    colunas = ["station_id", "last_reported", "num_bikes_available", "num_docks_available", "is_installed"]

    chunk = chunk[colunas].copy()

    chunk["is_installed"] = chunk["is_installed"].astype(int)
    chunk["timestamp_utc"] = chunk["last_reported"].dt.tz_localize("UTC")
    chunk["timestamp_local"] = chunk["timestamp_utc"].dt.tz_convert("Europe/Dublin")

    #sqlite3 nao aguenta fuso, entao vira string
    chunk["timestamp_utc"] = chunk["timestamp_utc"].astype(str)
    chunk["timestamp_local"] = chunk["timestamp_local"].astype(str)

    return chunk.drop(columns=["last_reported"])

## criar o banco e a tabela readings
funcoes: sqlite3.connect, connection.execute
cria a tabela antes pra escolher os tipos, em vez de deixar o pandas decidir

In [45]:
connection = sqlite3.connect("../data/dublinbikes.db")

connection.execute("DROP TABLE IF EXISTS readings")

connection.execute("""
    CREATE TABLE readings (
        station_id TEXT,
        timestamp_utc TEXT,
        timestamp_local TEXT,
        num_bikes_available INTEGER,
        num_docks_available INTEGER,
        is_installed INTEGER
    )
""")

## carga em pedacos
le os 6 arquivos de 200 mil linhas por vez e grava no banco
funcoes: pd.read_csv com chunksize, to_sql
chunksize evita carregar 3.4M de linhas de uma vez na memoria

In [46]:
total_rows = 0

for path in bike_files:
    for chunk in pd.read_csv(path, chunksize=10000, parse_dates=["last_reported"]):
        clean = clean_chunk(chunk)
        clean.to_sql("readings", connection, if_exists="append", index=False)
        total_rows += len(clean)
    print(path, "ok")

print("total:", total_rows)

../data/raw/bikes_2026_01.csv ok
../data/raw/bikes_2026_02.csv ok
../data/raw/bikes_2026_03.csv ok
../data/raw/bikes_2026_04.csv ok
../data/raw/bikes_2026_05.csv ok
../data/raw/bikes_2026_06.csv ok
total: 3412090
